# 05. Positional Encoding: откуда Transformer знает порядок слов?

Мы уже умеем превращать token ID в векторы с помощью Embedding. Но пока есть одна проблема: сам по себе вектор токена не сообщает модели, на каком месте предложения он встретился.

В этом блокноте разберём, зачем Transformer нужна информация о позициях и как её добавить к эмбеддингам.

## 1. Почему важен порядок слов?

Сравним два предложения:

- `the dog bites the man`
- `the man bites the dog`

В них почти один и тот же набор слов, но смысл различается. Значит, модели важно знать не только, какие токены встретились, но и где именно они находятся.

Обычный слой Embedding выдаёт вектор для каждого токена. Если токен `dog` встречается в двух разных местах, исходный embedding у него будет один и тот же. Поэтому к embedding добавляют информацию о позиции токена.

**Positional Encoding** — это вектор, который описывает позицию токена в последовательности. В классическом варианте эти векторы складываются с token embeddings.

## 2. Подготовим пример

Создадим простую последовательность из четырёх токенов. Для наглядности будем использовать небольшую размерность векторов — 8.

In [ ]:
import torch
import torch.nn as nn
import math

torch.manual_seed(42)

tokens = torch.tensor([[2, 3, 4, 5]])  # batch из одного предложения, 4 токена
vocab_size = 10
d_model = 8
max_length = 4

token_embedding = nn.Embedding(vocab_size, d_model)
x = token_embedding(tokens)

print("Форма token ID:", tokens.shape)
print("Форма embeddings:", x.shape)
print("Векторы токенов:\n", x)

Форма `x` — `(1, 4, 8)`:

- `1` — количество предложений в batch;
- `4` — количество токенов в предложении;
- `8` — размерность вектора каждого токена.

Нам нужно создать позиционную матрицу формы `(4, 8)`: один позиционный вектор для каждой из четырёх позиций.

## 3. Синусоидальное позиционное кодирование

В оригинальной статье Transformer предложен способ кодировать позиции с помощью синусов и косинусов разных частот.

Для позиции `pos` и пары координат с индексом `i` используются формулы:

`PE(pos, 2i) = sin(pos / 10000^(2i / d_model))`

`PE(pos, 2i + 1) = cos(pos / 10000^(2i / d_model))`

Не обязательно сразу запоминать формулы. Главное понять идею: каждой позиции соответствует свой набор чисел. В разных координатах значения меняются с разной частотой.

Сейчас построим такую матрицу в PyTorch.

In [ ]:
# Номера позиций: 0, 1, 2, 3
positions = torch.arange(max_length).unsqueeze(1)  # форма (4, 1)

# Индексы чётных координат: 0, 2, 4, 6
dimensions = torch.arange(0, d_model, 2)

# Частоты изменения синусов и косинусов
div_term = torch.exp(dimensions * (-math.log(10000.0) / d_model))

# Создаём матрицу позиционных векторов
pe = torch.zeros(max_length, d_model)
pe[:, 0::2] = torch.sin(positions * div_term)
pe[:, 1::2] = torch.cos(positions * div_term)

print("Позиционная матрица:\n", pe)
print("Форма positional encoding:", pe.shape)

### Как читать эту матрицу?

В ней 4 строки — позиции 0, 1, 2 и 3. В каждой строке 8 чисел — размерность позиционного вектора.

Посмотри на первую строку: для позиции 0 синусоидальные координаты равны 0, а косинусоидальные — 1. Для следующих позиций значения меняются.

Важна не отдельная координата, а вся комбинация значений: она позволяет модели различать позиции.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(9, 3))
plt.imshow(pe.numpy(), aspect="auto")
plt.xlabel("Координата вектора")
plt.ylabel("Позиция токена")
plt.title("Синусоидальные positional encodings")
plt.colorbar(label="Значение")
plt.show()

## 4. Добавляем позиционную информацию к Embedding

Теперь у нас есть две матрицы:

- `x`: эмбеддинги токенов, форма `(1, 4, 8)`;
- `pe`: позиционные векторы, форма `(4, 8)`.

Мы складываем их. PyTorch автоматически применяет broadcasting: одна и та же позиционная матрица добавляется к каждому примеру в batch.

То есть итоговый вектор токена содержит и информацию о самом токене, и информацию о его позиции.

In [ ]:
x_with_position = x + pe

print("Форма исходных embeddings:", x.shape)
print("Форма positional encoding:", pe.shape)
print("Форма результата:", x_with_position.shape)

print("\nEmbedding первого токена:\n", x[0, 0])
print("\nPositional encoding позиции 0:\n", pe[0])
print("\nИх сумма:\n", x_with_position[0, 0])

## 5. Всегда ли positional encoding синусоидальный?

Нет. Существует и другой распространённый подход: **обучаемые позиционные эмбеддинги**.

Вместо вычисления синусов и косинусов мы создаём ещё один слой `nn.Embedding`. Он хранит отдельный обучаемый вектор для каждой позиции. Во время обучения эти векторы меняются так же, как и token embeddings.

Для простой модели это можно реализовать так:

In [ ]:
position_ids = torch.arange(max_length).unsqueeze(0)  # [[0, 1, 2, 3]]
position_embedding = nn.Embedding(max_length, d_model)

learned_positions = position_embedding(position_ids)
x_with_learned_positions = x + learned_positions

print("Форма position IDs:", position_ids.shape)
print("Форма позиционных эмбеддингов:", learned_positions.shape)
print("Форма результата:", x_with_learned_positions.shape)

### Два подхода

| Синусоидальный | Обучаемый |
|---|---|
| Значения вычисляются по формуле | Значения являются параметрами модели |
| Не добавляет обучаемых параметров | Добавляет обучаемые параметры |
| Можно вычислить кодирование для новых позиций по формуле | Таблица обычно ограничена заданной максимальной длиной |

В нашем проекте важно понимать оба варианта. Для первого учебного Transformer мы выберем один подход и будем придерживаться его во всех экспериментах, чтобы сравнения были корректными.

## 6. Самостоятельные упражнения

1. Измени `max_length` на 6 и создай positional encoding для шести позиций. Как изменилась форма `pe`?
2. Измени `d_model` с 8 на 16 и пересоздай позиционную матрицу. Что изменилось?
3. Посмотри на строки `pe[0]`, `pe[1]`, `pe[2]`. Одинаковые ли у них значения? Почему?
4. Представь, что мы поменяли местами два токена в предложении. Изменятся ли их token embeddings? А итоговые векторы после добавления positional encoding?
5. Объясни своими словами, зачем Transformer добавляет позиционные векторы к эмбеддингам.

**Важно:** если меняешь `max_length` или `d_model`, перезапусти ячейки, которые создают embeddings и positional encoding, чтобы размеры совпадали.

## Что мы узнали

Теперь мы знаем, как Transformer получает информацию о порядке токенов: к token embeddings добавляются positional encodings.

У нас есть входная последовательность векторов, содержащая информацию и о токенах, и об их позициях.

Следующий шаг — **Self-Attention**. Там модель начнёт вычислять, какие токены важны друг для друга, используя три преобразования: Query, Key и Value.